# Google Colab — Fine-tuning de la CNN6 (modelo pulmonar, sonidos adventicios)

Entrena la CNN6 preentrenada de PANNs sobre los ciclos respiratorios de ICBHI
(detección de crackles / wheezes), con los **mismos 5 folds por paciente** que
todo el modelo pulmonar, para comparar directo contra v4 y contra el fine-tuning
parcial en CPU (v7). Detalle en `avances/17`.

**Qué hace cada cosa:**
- **Código**: se clona del repositorio de GitHub (siempre la última versión subida).
- **Datos**: un único archivo `colab_pulmonary_package.zip` en tu Google Drive
  (espectrogramas ya calculados + índice + folds + pesos de CNN6). Se copia una
  sola vez al disco local de la VM: así se evita el cuello de botella de leer
  miles de archivos desde Drive que hizo fallar el intento anterior (`avances/5`).
- **Resultados**: se guardan en tu Drive, época por época.

**Antes de correr:**
1. Subir el código nuevo a GitHub (`git push`): este notebook usa
   `desarrollo/src/pulmonary_adventitious_model/run_finetune_gpu.py`.
2. Subir `desarrollo/data/colab_pulmonary_package.zip` (263 MB, lo genera
   `pack_for_colab.py`) a la carpeta de Drive de la celda de configuración.
3. Activar la GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

**Si la sesión se corta** (Colab gratuito lo hace): volver a correr todas las
celdas. El entrenamiento retoma desde la última época guardada en Drive y no
repite los folds ya terminados.

In [ ]:
print("=== Montando Google Drive ===")
from google.colab import drive
drive.mount('/content/drive')

## Configuración

`VARIANT` elige qué se entrena (ver docstring de `run_finetune_gpu.py`):

| Variante | Qué se entrena | Épocas | Para qué |
|---|---|---|---|
| `A` | Bloque 4 + fc1 | 60 | El fine-tuning parcial de CPU (v7) con el doble de épocas |
| `B` | Bloques 3-4 + fc1, augmentation completa | 40 | Adaptar más la red, con datos más variados |
| `C` | Toda la red, learning rate más bajo, augmentation completa | 30 | El techo (mayor riesgo de sobreajuste) |

Conviene correrlas en orden: A, B, C. Cada una guarda sus resultados en su
propia carpeta (`v8_cnn6_ft_A`, `..._B`, `..._C`) dentro de `OUT_DIR`.

In [ ]:
from pathlib import Path

# Editá estas rutas si usás otra carpeta de Drive.
DRIVE_DIR   = Path("/content/drive/MyDrive/CEIA/colab_pulmonar")
PACKAGE_ZIP = DRIVE_DIR / "colab_pulmonary_package.zip"
OUT_DIR     = DRIVE_DIR / "resultados"

VARIANT = "A"   # "A", "B" o "C"

REPO_URL = "https://github.com/SantiagoBettig/deteccion_patologias_cardiopulmonares.git"
REPO_DIR = Path("/content/repo")
PKG_DIR  = Path("/content/pkg")

assert PACKAGE_ZIP.exists(), f"No se encontró el paquete de datos: {PACKAGE_ZIP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Paquete: {PACKAGE_ZIP} ({PACKAGE_ZIP.stat().st_size / 1e6:.0f} MB)")
print(f"✓ Resultados en: {OUT_DIR}")
print(f"✓ Variante: {VARIANT}")

In [ ]:
print("=== Verificando GPU ===")
import torch
print("CUDA disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución"
print("Dispositivo:", torch.cuda.get_device_name(0))

In [ ]:
print("=== Clonando / actualizando el repositorio ===")
import subprocess
if (REPO_DIR / ".git").exists():
    print(subprocess.run(["git", "-C", str(REPO_DIR), "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "desarrollo" / "src"
assert (SRC_DIR / "pulmonary_adventitious_model" / "run_finetune_gpu.py").exists(),     "El repositorio no tiene run_finetune_gpu.py: ¿hiciste git push del código nuevo?"
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
print("=== Dependencias ===")
# torch, numpy, pandas, scikit-learn, matplotlib, librosa y soundfile ya vienen en Colab.
# No se instala requirements.txt para no arriesgar reemplazar el torch con GPU de Colab.
import librosa, soundfile, sklearn, pandas, numpy, matplotlib
print("✓ librosa", librosa.__version__, "| torch", torch.__version__)

In [ ]:
print("=== Copiando el paquete de datos al disco local de la VM ===")
import shutil, time, zipfile
if not (PKG_DIR / "logmel_panns32k.npy").exists():
    t0 = time.time()
    local_zip = Path("/content/package.zip")
    shutil.copy2(PACKAGE_ZIP, local_zip)
    zipfile.ZipFile(local_zip).extractall(PKG_DIR)
    local_zip.unlink()
    print(f"✓ Copiado y descomprimido en {time.time() - t0:.0f} s")
print(sorted(p.name for p in PKG_DIR.iterdir()))

## Entrenamiento

Imprime una línea por época (loss de train, AUC de validación, segundos) y el
resultado de test al terminar cada fold. Si la sesión se corta, volver a correr
desde arriba: retoma solo.

In [ ]:
!cd {SRC_DIR} && python -u pulmonary_adventitious_model/run_finetune_gpu.py     --package-dir {PKG_DIR} --out-dir {OUT_DIR} --variant {VARIANT}

## Después de entrenar

Los resultados quedan en `OUT_DIR/v8_cnn6_ft_<variante>/` en tu Drive:
`summary.csv`, `per_fold.csv`, `oof_predictions.csv`, `metrics_by_equipment.csv`,
`metrics_by_age_group.csv`, `confusion_matrix_4class.png`, `training_curves.png`
y los checkpoints `fold{i}_best.pt`.

Para analizarlos localmente, copiá esa carpeta a
`desarrollo/reports/pulmonary_adventitious/` (los checkpoints `.pt` pueden ir a
`desarrollo/src/pulmonary_adventitious_model/checkpoints/`).